In [1]:
import pandas as pd
import numpy as np
import ast

import xml.etree.ElementTree as ET
from IPython.display import display, HTML
from sklearn.model_selection import GroupShuffleSplit

PATH_ACTIONS                        = "../../Data/Unprocessed/actions.xml"
PATH_ACTIONS_FILTERED               = "../../Data/Processed/actions_filtered.csv"

PATH_CLIPS                          = "../../Data/Videos/Clips/"
PATH_ROI                            = "../../Data/Unprocessed/roi.csv"

PATH_KEYPOINTS                      = "../../Data/Unprocessed/keypoints.csv"
PATH_METRICS                        = "../../Data/Unprocessed/metrics.csv"

PATH_CLS_DATA                       = "../../Data/Processed/cls_data.csv"
PATH_LABEL_MAP                      = "../../Data/label_map.csv"

PATH_ROI_TRAIN                      = "../../Data/Processed/roi_train.csv"
PATH_ROI_TEST                       = "../../Data/Processed/roi_test.csv"

WINDOW_SIZE                         = 4

In [2]:
def parse_xml(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    records = []

    # --------------------------------------------------------------------------
    # STEP 1: Extract task-level metadata (id → source filename)
    # --------------------------------------------------------------------------
    task_sources = {}
    for task in root.findall(".//meta/project/tasks/task"):
        task_id = task.findtext("id")
        source = task.findtext("source")
        name = task.findtext("name")

        if task_id:
            task_sources[task_id] = {
                "source": source,
                "name": name
            }

    # --------------------------------------------------------------------------
    # STEP 2: Parse all tracks and link to the correct task
    # --------------------------------------------------------------------------
    for track in root.findall("track"):
        task_id = track.get("task_id")
        label = track.get("label")

        # Match this track to its source video via task_id
        task_info = task_sources.get(task_id, {})
        source = task_info.get("source", "unknown")
        task_name = task_info.get("name", "unknown")

        # ---- Extract per-frame boxes and attributes ----
        for box in track.findall("box"):
            frame = int(box.get("frame"))

            if label == "Strip":
                xtl = float(box.get("xtl"))
                ytl = float(box.get("ytl"))
                xbr = float(box.get("xbr"))
                ybr = float(box.get("ybr"))

                frame_attrs = {
                    attr.get("name"): attr.text.strip() if attr.text else ""
                    for attr in box.findall("attribute")
                }

                records.append({
                    "task_name": task_name,
                    "source": source,
                    "frame": frame,
                    "xtl": xtl,
                    "ytl": ytl,
                    "xbr": xbr,
                    "ybr": ybr,
                    **frame_attrs
                })

    return pd.DataFrame(records)

In [3]:
def combine_frames(df):
    df["clip_number"] = df["file"].str.extract(r"^(\d+)").astype(int)
    df["frame"] = df.groupby(["file"])["frame"].transform(lambda x: x - x.min())

    # Sort by hierarchy including numeric clip number
    df = df.sort_values(["file", "fencer", "frame"]).reset_index(drop=True)

    # Columns to group by for hierarchy
    group_cols = ["file", "fencer"]
    results = []

    # Iterate over groups
    for _, grp in df.groupby(group_cols):
        grp = grp.sort_values("frame").reset_index(drop=True)
        
        # Use shift/cumsum to identify consecutive runs
        grp["run"] = (grp["action"] != grp["action"].shift()).cumsum()
        
        # Aggregate start/end frames per run
        run_df = grp.groupby(["run", "action"]).agg(
            start_frame=("frame", "min"),
            end_frame=("frame", "max")
        ).reset_index(drop=False)
        
        # Add hierarchy columns
        run_df["file"] = grp["file"].iloc[0]
        run_df["fencer"] = grp["fencer"].iloc[0]
        
        # Keep only desired columns
        run_df = run_df[["file", "fencer", "action", "start_frame", "end_frame"]]
        results.append(run_df)

    # Combine all groups
    return pd.concat(results, ignore_index=True)

In [4]:
def parse_annotations(path_annotations):
    df = parse_xml(path_annotations)

    df["task_name"] = df["task_name"].str.replace("Bout ", "", regex=False)
    df["task_name"] = df["task_name"].replace("Test Upload", "1")
    df["file"] = df["task_name"] + "/" + df["source"]

    df_roi = df[["file", "frame", "xtl", "ytl", "xbr", "ybr"]]
    df_roi.to_csv(PATH_ROI)

    df = df.melt(
        id_vars=["file", "frame"],
        value_vars=["Fencer_L", "Fencer_R"],
        var_name="fencer",
        value_name="action"
    )

    df["fencer"] = df["fencer"].map({
        "Fencer_L": "LEFT",
        "Fencer_R": "RIGHT"
    })

    cols = ["file", "frame", "fencer", "action"]
    df = df[cols]
    df["frame"] = df.groupby(["file"])["frame"].transform(lambda x: x - x.min())

    return df

    return #combine_frames(df[cols])

In [5]:
def show_stats(df):
    def display_actions(df):
        html_str = ""
        title = "<h4>Actions</h4>"
        html_str += f"""
        <div style="display: inline-block; vertical-align: top; margin-right: 30px;">
            {title}
            {df.to_html(index=False)}
        </div>
        """
        display(HTML(html_str))

    def get_stats(df):
        df = df.copy()
        df["duration"] = df["end_frame"] - df["start_frame"] + 1
        stats = (
            df.groupby("action")["duration"]
            .agg(["min", "max", "mean", "std"])
            .reset_index()
        )

        stats["count"] = df["action"].value_counts().reindex(stats["action"]).values
        return stats.sort_values(by="count", ascending=False)

    total_actions = len(df)
    stats = get_stats(df)

    display_actions(stats)

    print(f"\nTotal Actions: {total_actions}")
    print("Total classes: ", df["action"].nunique())
    print("")

In [16]:
def create_sliding_windows(df,window_size=4,stride=1):
    window_rows = []
    window_counter = 0

    # Process each video separately
    for file_name, group in df.groupby(["file"]):
        group = group.sort_values(["frame"]).reset_index(drop=True)

        n_frames = len(group)
        if n_frames < window_size:
            continue

        for start in range(0, n_frames - window_size + 1, stride):

            end = start + window_size
            window_slice = group.iloc[start:end].copy()

            window_counter += 1
            window_slice["window_id"] = window_counter
            window_slice["left_action"] = window_slice.iloc[window_size - 1]["left_action"]
            window_slice["right_action"] = window_slice.iloc[window_size - 1]["right_action"]

            window_rows.append(window_slice)

    if not window_rows:
        return pd.DataFrame()

    return pd.concat(window_rows, ignore_index=True)

In [32]:
def explode_keypoints(df):
    df = df.copy()
    # Expand each tuple into separate x,y columns
    exploded_left = df["left_keypoints"].apply(
        lambda kp: [coord for point in kp for coord in point]
    )

    exploded_right = df["right_keypoints"].apply(
        lambda kp: [coord for point in kp for coord in point]
    )

    # Create column names: x0, y0, x1, y1, ...
    num_points = len(df.iloc[0]["left_keypoints"])
    left_cols = [f"xl{i}" for i in range(num_points)] + [f"yl{i}" for i in range(num_points)]

    # Create column names: x0, y0, x1, y1, ...
    num_points = len(df.iloc[0]["right_keypoints"])
    right_cols = [f"xr{i}" for i in range(num_points)] + [f"yr{i}" for i in range(num_points)]
    
    left_df = pd.DataFrame(exploded_left.tolist(), columns=left_cols)
    right_df = pd.DataFrame(exploded_right.tolist(), columns=right_cols)

    new_df = pd.concat([left_df, right_df], axis=1)

    return pd.concat([df.drop(columns=["left_keypoints", "right_keypoints"]), new_df], axis=1)

In [7]:
df = parse_annotations(PATH_ACTIONS)

df.sort_values(by=["file", "fencer", "frame"], inplace=True)
df = df.reset_index(drop=True)

#df["duration"] = df["end_frame"] - df["start_frame"] + 1
#df["action_id"] = df.index.astype(int)

df = df[["file", "fencer", "frame", "action"]]

#show_stats(df)
print("")
print(df)

#df.to_csv(PATH_ACTIONS_FILTERED, index=False)


                file fencer  frame        action
0      1/10_Left.mp4   LEFT      0     NO_ACTION
1      1/10_Left.mp4   LEFT      1     NO_ACTION
2      1/10_Left.mp4   LEFT      2     NO_ACTION
3      1/10_Left.mp4   LEFT      3     NO_ACTION
4      1/10_Left.mp4   LEFT      4     NO_ACTION
...              ...    ...    ...           ...
16823   6/9_Left.mp4  RIGHT     40  SHORT_ATTACK
16824   6/9_Left.mp4  RIGHT     41  SHORT_ATTACK
16825   6/9_Left.mp4  RIGHT     42  SHORT_ATTACK
16826   6/9_Left.mp4  RIGHT     43  SHORT_ATTACK
16827   6/9_Left.mp4  RIGHT     44  SHORT_ATTACK

[16828 rows x 4 columns]


In [92]:
unique_labels = sorted(df["action"].unique())

labels = pd.DataFrame(unique_labels, columns=["action"])
labels["id"] = labels.index

labels.rename(columns={"action": "label", "action_id": "weight"}, inplace=True)
labels.to_csv(PATH_LABEL_MAP, index=False)

print(labels)

          label  id
0     DIST_PULL   0
1   LONG_ATTACK   1
2     NO_ACTION   2
3         PARRY   3
4  SHORT_ATTACK   4


In [93]:
df_metrics = pd.read_csv(PATH_METRICS)

total_expected_frames = df_metrics["expected"].sum()
total_actual_frames = df_metrics["actual"].sum()
total_coverage = total_actual_frames / total_expected_frames

print(df_metrics[df_metrics["expected"] != df_metrics["actual"]])
print("")

print("Expected frames: ", total_expected_frames)
print("Actual frames:   ", total_actual_frames)
print(f"Coverage:         {total_coverage*100:.2f}%")

    fencer  action_id  start_frame  end_frame  expected  actual   coverage  \
275  RIGHT        997           31         40        10       9  90.000000   
619  RIGHT        768           27         32         6       5  83.333333   

               file  
275  6/20_Right.mp4  
619   5/15_Left.mp4  

Expected frames:  16828
Actual frames:    16826
Coverage:         99.99%


In [9]:
df_keypoints = pd.read_csv(PATH_KEYPOINTS)

df_keypoints["keypoints"] = df_keypoints["keypoints"].apply(
    lambda x: [tuple(p) for p in ast.literal_eval(x)] if isinstance(x, str) else x
)

df_merged = df_keypoints.merge(df, on=["file", "fencer", "frame"], how="left")

df_merged = df_merged[["file", "fencer", "frame", "action", "keypoints"]].reset_index(drop=True)
print(df_merged)

                file fencer  frame        action  \
0      1/10_Left.mp4   LEFT      0     NO_ACTION   
1      1/10_Left.mp4   LEFT      1     NO_ACTION   
2      1/10_Left.mp4   LEFT      2     NO_ACTION   
3      1/10_Left.mp4   LEFT      3     NO_ACTION   
4      1/10_Left.mp4   LEFT      4     NO_ACTION   
...              ...    ...    ...           ...   
16823   6/9_Left.mp4  RIGHT     40  SHORT_ATTACK   
16824   6/9_Left.mp4  RIGHT     41  SHORT_ATTACK   
16825   6/9_Left.mp4  RIGHT     42  SHORT_ATTACK   
16826   6/9_Left.mp4  RIGHT     43  SHORT_ATTACK   
16827   6/9_Left.mp4  RIGHT     44  SHORT_ATTACK   

                                               keypoints  
0      [(651.2871704101562, 605.3470458984375), (654....  
1      [(652.6170654296875, 606.501953125), (655.7320...  
2      [(654.3292236328125, 608.1226806640625), (657....  
3      [(658.3043823242188, 607.803466796875), (661.1...  
4      [(664.1746826171875, 605.3037719726562), (665....  
...                  

In [15]:
df_left = df_merged[df_merged["fencer"] == "LEFT"].copy()
df_right = df_merged[df_merged["fencer"] == "RIGHT"].copy()

df_left.rename(columns={"keypoints": "left_keypoints", "action": "left_action"}, inplace=True)
df_right.rename(columns={"keypoints": "right_keypoints", "action": "right_action"}, inplace=True)

df_left.drop(columns=["fencer"], inplace=True)
df_right.drop(columns=["fencer"], inplace=True)

df_combined = df_left.merge(df_right, on=["file", "frame"], how="left")
print(df_combined)

               file  frame left_action  \
0     1/10_Left.mp4      0   NO_ACTION   
1     1/10_Left.mp4      1   NO_ACTION   
2     1/10_Left.mp4      2   NO_ACTION   
3     1/10_Left.mp4      3   NO_ACTION   
4     1/10_Left.mp4      4   NO_ACTION   
...             ...    ...         ...   
8409   6/9_Left.mp4     40   DIST_PULL   
8410   6/9_Left.mp4     41   DIST_PULL   
8411   6/9_Left.mp4     42   DIST_PULL   
8412   6/9_Left.mp4     43   DIST_PULL   
8413   6/9_Left.mp4     44   DIST_PULL   

                                         left_keypoints  right_action  \
0     [(651.2871704101562, 605.3470458984375), (654....     NO_ACTION   
1     [(652.6170654296875, 606.501953125), (655.7320...     NO_ACTION   
2     [(654.3292236328125, 608.1226806640625), (657....     NO_ACTION   
3     [(658.3043823242188, 607.803466796875), (661.1...     NO_ACTION   
4     [(664.1746826171875, 605.3037719726562), (665....     NO_ACTION   
...                                                 ...  

In [33]:
df_windowed = create_sliding_windows(df_combined, window_size=WINDOW_SIZE)

print("Number of action snippets: ", df_windowed["window_id"].nunique())
print("")
print(df_windowed)

Number of action snippets:  7967

                file  frame left_action  \
0      1/10_Left.mp4      0   NO_ACTION   
1      1/10_Left.mp4      1   NO_ACTION   
2      1/10_Left.mp4      2   NO_ACTION   
3      1/10_Left.mp4      3   NO_ACTION   
4      1/10_Left.mp4      1   NO_ACTION   
...              ...    ...         ...   
31863   6/9_Left.mp4     43   DIST_PULL   
31864   6/9_Left.mp4     41   DIST_PULL   
31865   6/9_Left.mp4     42   DIST_PULL   
31866   6/9_Left.mp4     43   DIST_PULL   
31867   6/9_Left.mp4     44   DIST_PULL   

                                          left_keypoints  right_action  \
0      [(651.2871704101562, 605.3470458984375), (654....     NO_ACTION   
1      [(652.6170654296875, 606.501953125), (655.7320...     NO_ACTION   
2      [(654.3292236328125, 608.1226806640625), (657....     NO_ACTION   
3      [(658.3043823242188, 607.803466796875), (661.1...     NO_ACTION   
4      [(652.6170654296875, 606.501953125), (655.7320...     NO_ACTION   
...  

In [35]:
df_windowed.sort_values(["file", "window_id", "frame"], inplace=True)

df_exploded = explode_keypoints(df_windowed)
print(df_exploded)
df_exploded.to_csv(PATH_CLS_DATA, index=False)

                file  frame left_action  right_action  window_id         xl0  \
0      1/10_Left.mp4      0   NO_ACTION     NO_ACTION          1  651.287170   
1      1/10_Left.mp4      1   NO_ACTION     NO_ACTION          1  652.617065   
2      1/10_Left.mp4      2   NO_ACTION     NO_ACTION          1  654.329224   
3      1/10_Left.mp4      3   NO_ACTION     NO_ACTION          1  658.304382   
4      1/10_Left.mp4      1   NO_ACTION     NO_ACTION          2  652.617065   
...              ...    ...         ...           ...        ...         ...   
31863   6/9_Left.mp4     43   DIST_PULL  SHORT_ATTACK       7966  809.526611   
31864   6/9_Left.mp4     41   DIST_PULL  SHORT_ATTACK       7967  830.304749   
31865   6/9_Left.mp4     42   DIST_PULL  SHORT_ATTACK       7967  816.049316   
31866   6/9_Left.mp4     43   DIST_PULL  SHORT_ATTACK       7967  809.526611   
31867   6/9_Left.mp4     44   DIST_PULL  SHORT_ATTACK       7967  798.216919   

              xl1         xl2         x

In [97]:
df_roi = pd.read_csv(PATH_ROI)

gss_roi = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    random_state=42
)

train_idx, test_idx = next(
    gss_roi.split(df_roi, groups=df_roi['file'])
)

df_train_roi = df_roi.iloc[train_idx]
df_test_roi  = df_roi.iloc[test_idx]

df_train_roi.to_csv(PATH_ROI_TRAIN, index=False)
df_test_roi.to_csv(PATH_ROI_TEST, index=False)